In [ ]:
#────────────────────────────────────────────────────
# Instalace knihoven
#────────────────────────────────────────────────────

!pip install -q planetary-computer pystac-client stackstac \
              rioxarray geopandas shapely pyproj dask[complete]

In [ ]:
#────────────────────────────────────────────────────
# Import
#────────────────────────────────────────────────────

import os
import gc
import shutil

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import planetary_computer as pc
import stackstac
import rioxarray

from pystac_client import Client
from rasterio.enums import Resampling
from dask.diagnostics import ProgressBar
from google.colab import drive

In [ ]:
#────────────────────────────────────────────────────
# Připojení Google Disku
#────────────────────────────────────────────────────

drive.mount('/content/drive')

In [ ]:
#────────────────────────────────────────────────────
# Parametry
#────────────────────────────────────────────────────

months_config = [
    {"name": "may",  "start": "2026-05-01", "end": "2026-05-31"},
    {"name": "june", "start": "2026-06-01", "end": "2026-06-30"},
    {"name": "july", "start": "2026-07-01", "end": "2026-07-31"},
    {"name": "aug",  "start": "2026-08-01", "end": "2026-08-31"},
]

shp_path = "/content/tiles_czechia.shp"
output_dir = "/content/drive/MyDrive/S1_exporty"

os.makedirs(output_dir, exist_ok=True)

chunk_size = 1024
max_retries = 3

epsg = 32633
resolution = 10
polarization = "vh"

In [ ]:
#────────────────────────────────────────────────────
# Funkce
#────────────────────────────────────────────────────

def search_s1_rtc(
    bbox,
    start_date,
    end_date,
    orbit,
    polarization="vh",
):
    search = catalog.search(
        collections=["sentinel-1-rtc"],
        bbox=bbox,
        datetime=f"{start_date}/{end_date}",
    )

    items = list(search.items())

    # Explicitní kontrola orbity.
    items = [
        item
        for item in items
        if item.properties.get("sat:orbit_state") == orbit
    ]

    # Použijeme pouze scény obsahující požadovanou polarizaci.
    items = [
        item
        for item in items
        if polarization in item.assets
    ]

    print(
        f"{start_date} - {end_date}: "
        f"{len(items)} scén "
        f"({orbit}, {polarization.upper()})"
    )

    return items


def process_stack(
    signed_items,
    gdf,
    polarization="vh",
    epsg=32633,
    resolution=10,
    chunk_size=1024,
    band_name=None,
):
    print(
        f"Start stack processing — "
        f"POL: {polarization.upper()}, "
        f"EPSG: {epsg}, "
        f"Resolution: {resolution} m, "
        f"Chunk: {chunk_size}"
    )

    tile_bounds = gdf.to_crs(epsg).total_bounds.tolist()

    da = stackstac.stack(
        signed_items,
        assets=[polarization],
        resolution=resolution,
        epsg=epsg,
        bounds=tile_bounds,
        dtype="float32",
        fill_value=np.float32(np.nan),
        rescale=False,
        resampling=Resampling.bilinear,
    ).rename(band_name).squeeze()

    da = da.chunk({
        "time": min(8, da.sizes["time"]),
        "y": chunk_size,
        "x": chunk_size,
    })

    print(
        f"Dataset velikost: {da.shape}, "
        f"chunks: {da.chunks}"
    )

    # Oříznutí na polygon tile.
    da_clip = da.rio.clip(
        gdf.to_crs(epsg).geometry,
        epsg,
        drop=False,
    )

    # Časová dimenze v jednom chunku.
    da_clip = da_clip.chunk({"time": -1})

    # Medián počítáme v lineární škále.
    da_filt = da_clip.median(
        dim="time",
        skipna=True,
    )

    # Převod z lineární škály do dB.
    # Neplatné/nulové hodnoty zůstávají jako NaN.
    da_filt_db = 10 * np.log10(
        da_filt.where(da_filt > 0)
    )

    print(
        f"Hotovo! Výstupní shape: "
        f"{da_filt_db.shape}"
    )

    return da_filt_db


def save_raster(da, out_fp):
    da.rio.to_raster(
        out_fp,
        driver="GTiff",
        compress="deflate",
        predictor=1,
        BIGTIFF="IF_SAFER",
    )

    print(
        f"TIFF byl uložen: {out_fp}"
    )

In [ ]:
#────────────────────────────────────────────────────
# Načtení SHP + vizualizace
#────────────────────────────────────────────────────

gdf = gpd.read_file(
    shp_path
).to_crs("EPSG:4326")

fig, ax = plt.subplots()

gdf.plot(
    ax=ax,
    facecolor="none",
    edgecolor="green",
)

plt.show()


In [ ]:
#────────────────────────────────────────────────────
# Připojení ke STAC API
#────────────────────────────────────────────────────

catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

In [ ]:
#────────────────────────────────────────────────────
# ITERACE přes měsíce, orbity a tiles
#────────────────────────────────────────────────────

failed_runs = []
empty_runs = []


for m in months_config:

    month = m["name"]
    start_date = m["start"]
    end_date = m["end"]

    print(
        f"\nZAHÁJENO ZPRACOVÁNÍ MĚSÍCE: "
        f"{month.upper()} "
        f"({start_date} až {end_date})"
    )

    for orbit_direction in [
        "ascending",
        "descending",
    ]:

        orbit_short = (
            "asc"
            if orbit_direction == "ascending"
            else "desc"
        )

        sub_dir_name = (
            f"S1_{month}_{orbit_short}"
        )

        sub_dir_path = os.path.join(
            output_dir,
            sub_dir_name,
        )

        os.makedirs(
            sub_dir_path,
            exist_ok=True,
        )

        print(
            f"ORBIT: {orbit_direction.upper()} | "
            f"MĚSÍC: {month.upper()} | "
            f"SLOŽKA: {sub_dir_name}"
        )

        for idx, row in gdf.iterrows():

            tile_num = idx + 1

            filename = (
                f"S1_{month}_{orbit_short}_"
                f"tile{tile_num:02d}.tif"
            )

            out_fp = os.path.join(
                sub_dir_path,
                filename,
            )

            local_fp = os.path.join(
                "/content",
                f"temp_{filename}",
            )

            # Pokud výstup již existuje, není potřeba
            # tile znovu počítat.
            if os.path.exists(out_fp):

                print(
                    f"Tile {tile_num}/{len(gdf)} "
                    f"({sub_dir_name}) již existuje "
                    f"na Disku. Přeskakuji."
                )

                continue

            print(
                f"\nZpracovávám tile "
                f"{tile_num}/{len(gdf)} "
                f"({month.upper()}, "
                f"{orbit_direction.upper()})"
            )

            attempt = 0
            success = False

            while attempt < max_retries and not success:

                try:

                    tile_gdf = gpd.GeoDataFrame(
                        geometry=[row.geometry],
                        crs=gdf.crs,
                    )

                    tile_bbox = (
                        tile_gdf
                        .total_bounds
                        .tolist()
                    )

                    # Vyhledání scén.
                    items_orbit = search_s1_rtc(
                        tile_bbox,
                        start_date,
                        end_date,
                        orbit_direction,
                        polarization=polarization,
                    )

                    # Žádná dostupná data.
                    if not items_orbit:

                        print(
                            f"Žádné scény pro tile "
                            f"{tile_num} "
                            f"({sub_dir_name}). "
                            f"Přeskakuji."
                        )

                        empty_runs.append(
                            f"{sub_dir_name}_tile"
                            f"{tile_num:02d}"
                        )

                        success = True
                        break

                    # Podepsání URL assetů.
                    signed_items = [
                        pc.sign(item).to_dict()
                        for item in items_orbit
                    ]

                    # Stack + zpracování.
                    da_orbit = process_stack(
                        signed_items=signed_items,
                        gdf=tile_gdf,
                        polarization=polarization,
                        epsg=epsg,
                        resolution=resolution,
                        chunk_size=chunk_size,
                        band_name=f"{month}_{orbit_short}",
                    )

                    # Výpočet Dasku.
                    with ProgressBar():

                        da_orbit_local = (
                            da_orbit.compute()
                        )

                    # Uložení lokálně.
                    save_raster(
                        da_orbit_local,
                        local_fp,
                    )

                    # Přesun na Google Drive.
                    shutil.move(
                        local_fp,
                        out_fp,
                    )

                    # Uvolnění paměti.
                    del da_orbit
                    del da_orbit_local

                    gc.collect()

                    print(
                        f"Tile {tile_num} uložena do: "
                        f"{sub_dir_name}/{filename}"
                    )

                    success = True

                except Exception as e:

                    attempt += 1

                    print(
                        f"Chyba při zpracování tile "
                        f"{tile_num}, "
                        f"pokus {attempt}/{max_retries} "
                        f"→ {e}"
                    )

                    # Odstranění případného neúplného
                    # lokálního souboru.
                    if os.path.exists(local_fp):

                        os.remove(local_fp)

                    gc.collect()

                    if attempt < max_retries:

                        print(
                            "Opakuji pokus..."
                        )

                    else:

                        print(
                            "Tile selhala po "
                            "maximálním počtu pokusů."
                        )

                        failed_runs.append(
                            f"{sub_dir_name}_tile"
                            f"{tile_num:02d}"
                        )


#────────────────────────────────────────────────────
# Závěrečná zpráva
#────────────────────────────────────────────────────

print(
    "\nZÁVĚREČNÁ ZPRÁVA O ZPRACOVÁNÍ"
)


if failed_runs:

    print(
        f"Selhané tiles: {failed_runs}"
    )

else:

    print(
        "Žádná tile neselhala."
    )


if empty_runs:

    print(
        f"Tiles bez dostupných scén: "
        f"{empty_runs}"
    )

else:

    print(
        "Všechny požadované tiles měly "
        "alespoň jednu scénu."
    )